# Seamless widgets

Run all cells with the seamless1 Python kernel. In JupyterLab, drag either slider and confirm that the sum and output update after rerunning the compute cell. The automated assertions verify kernel transport and output data; visible frontend rendering needs manual inspection.

In [ ]:
import ipywidgets as widgets
from IPython.display import display
from seamless import Cell
from seamless_workflow import Context
from seamless_workflow.jupyter import traitlet, output

def add(a, b):
    return a + b

ctx = Context()
ctx.a = Cell(celltype="int")
ctx.a.set(1)
ctx.b = Cell(celltype="int")
ctx.b.set(2)
a = widgets.IntSlider(value=1, description="a")
b = widgets.IntSlider(value=2, description="b")
from tempfile import TemporaryDirectory
from pathlib import Path
workspace = TemporaryDirectory(prefix="seamless-widgets-")
a_path = Path(workspace.name) / "a.txt"
ctx.a.mount(a_path, mode="rw", authority="cell")
traitlet(ctx.a).link(a)
traitlet(ctx.b).link(b)
ctx.tf = add
ctx.tf.pins.a = ctx.a
ctx.tf.pins.b = ctx.b
ctx.c = ctx.tf.result
out = output(ctx.c)
display(a, b, out)

In [ ]:
a.value = 7
ctx.mounts.sync(timeout=10)
assert ctx.a.value == 7

ctx.a = 3
ctx.mounts.sync(timeout=10)
assert a.value == 3

In [ ]:
ctx.mounts.sync(timeout=10)
ctx.compute(timeout=30)
ctx.mounts.sync(timeout=10)
assert ctx.c.value == ctx.a.value + ctx.b.value
assert out.output_instance.outputs
assert str(ctx.c.value) in str(out.output_instance.outputs)

The following blocking lifecycle calls must finish while the kernel thread is busy. They exercise transport writes without scheduling work onto the kernel event loop.

In [ ]:
assert a_path.read_text().strip() == "3"
del ctx.a.mount
a.value = 9
ctx.mounts.sync(timeout=10)
assert ctx.a.value == 9
assert a_path.read_text().strip() == "3"
ctx.a.mount(a_path, mode="rw", authority="cell")
ctx.mounts.sync(timeout=10)
assert a_path.read_text().strip() == "9"
assert traitlet(ctx.a).value == 9

In [ ]:
ctx.close()
a.value = 10
b.value = 11
print("Widget kernel transport and lifecycle checks passed")
workspace.cleanup()